# Mine RESPONSE.csv Commit Dates

This notebook loads `../../data/baseline/RESPONSE.csv`, reads the `COMMIT` column, fetches each GitHub commit link, extracts the commit date/time, and writes a new CSV with the mined date added.

Paste a GitHub token in the first code cell before running. The token is only stored in the notebook runtime unless you save it in the notebook file.

In [8]:
from pathlib import Path
import json
import re
import time
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime

import pandas as pd
import requests

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

# Optional. Leave blank for public GitHub commit links.
# Paste a token only if some commit links are private or GitHub blocks unauthenticated requests.
GITHUB_TOKEN = ""

INPUT_CSV = Path("../../data/baseline/RESPONSE.csv")
OUTPUT_CSV = INPUT_CSV.with_name("RESPONSE_with_commit_dates.csv")
FAILURES_CSV = INPUT_CSV.with_name("RESPONSE_commit_date_failures.csv")
CACHE_JSON = INPUT_CSV.with_name("github_commit_page_date_cache.json")

COMMIT_COLUMN = "COMMIT"

# If COMMIT contains only a SHA, this column provides the GitHub repo.
# Accepted repo formats include owner/repo, https://github.com/owner/repo, or git@github.com:owner/repo.git.
REPO_COLUMN = "DOWNSTREAM REPO"

DATE_COLUMN = "COMMIT_DATE"
REQUEST_SLEEP_SECONDS = 1.0
REQUEST_TIMEOUT_SECONDS = 30
MAX_RETRIES = 3
RATE_LIMIT_BUFFER_SECONDS = 30
RETRY_FAILED_CACHE = True

In [9]:
if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Could not find {INPUT_CSV.resolve()}")

df = pd.read_csv(INPUT_CSV)

if COMMIT_COLUMN not in df.columns:
    raise ValueError(f"{INPUT_CSV} must contain a {COMMIT_COLUMN!r} column. Found: {list(df.columns)}")

if REPO_COLUMN is not None and REPO_COLUMN not in df.columns:
    raise ValueError(f"REPO_COLUMN={REPO_COLUMN!r} is not in the CSV columns: {list(df.columns)}")

print(f"Loaded {len(df):,} rows from {INPUT_CSV}")
display(df.head())

Loaded 3,312 rows from ../../data/baseline/RESPONSE.csv


,CVE,UPSTREAM G:A:V,DOWNSTREAM G:A:V,DOWNSTREAM REPO,COMMIT
0,CVE-2021-39154,com.thoughtworks.xstream:xstream,ne,dbmdz/digitalcollections-model,https://github.com/dbmdz/digitalcollections-mo...
1,CVE-2021-37714,org.jsoup:jsoup,com.github.btheu.estivate:estivate,btheu/estivate,https://github.com/btheu/estivate/commit/22adf...
2,CVE-2021-37714,org.jsoup:jsoup,com.jcabi:jcabi-http,jcabi/jcabi-http,https://github.com/jcabi/jcabi-http/commit/1a2...
3,CVE-2021-37714,org.jsoup:jsoup,in.ashwanthkumar:gocd-java-client,ashwanthkumar/gocd-java-client,https://github.com/ashwanthkumar/gocd-java-cli...
4,CVE-2021-37714,org.jsoup:jsoup,tech.grasshopper:pdfextentreporter,grasshopper7/pdfextentreporter,https://github.com/grasshopper7/pdfextentrepor...


In [10]:
GITHUB_COMMIT_URL_RE = re.compile(
    r"github\.com/(?P<owner>[^/]+)/(?P<repo>[^/]+)/commit/(?P<sha>[0-9a-fA-F]{7,40})"
)
SHA_RE = re.compile(r"^[0-9a-fA-F]{7,40}$")


def normalize_repo(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    if not text:
        return None

    text = text.removesuffix(".git")

    https_match = re.search(r"github\.com[:/](?P<owner>[^/]+)/(?P<repo>[^/#?]+)", text)
    if https_match:
        return f"{https_match.group('owner')}/{https_match.group('repo').removesuffix('.git')}"

    owner_repo_match = re.fullmatch(r"(?P<owner>[^/\s]+)/(?P<repo>[^/\s]+)", text)
    if owner_repo_match:
        return f"{owner_repo_match.group('owner')}/{owner_repo_match.group('repo').removesuffix('.git')}"

    return None


def parse_commit_reference(commit_value, repo_value=None):
    if pd.isna(commit_value):
        return None, None, None

    commit_text = str(commit_value).strip()
    if not commit_text:
        return None, None, None

    url_match = GITHUB_COMMIT_URL_RE.search(commit_text)
    if url_match:
        repo = f"{url_match.group('owner')}/{url_match.group('repo').removesuffix('.git')}"
        sha = url_match.group("sha")
        return repo, sha, f"https://github.com/{repo}/commit/{sha}"

    if SHA_RE.fullmatch(commit_text):
        repo = normalize_repo(repo_value)
        commit_url = f"https://github.com/{repo}/commit/{commit_text}" if repo else None
        return repo, commit_text, commit_url

    return normalize_repo(repo_value), None, None


parsed = df.apply(
    lambda row: parse_commit_reference(
        row[COMMIT_COLUMN], row[REPO_COLUMN] if REPO_COLUMN is not None else None
    ),
    axis=1,
)

df["_github_repo"] = [repo for repo, sha, commit_url in parsed]
df["_commit_sha"] = [sha for repo, sha, commit_url in parsed]
df["_commit_url"] = [commit_url for repo, sha, commit_url in parsed]
df["_parse_error"] = None
df.loc[df["_github_repo"].isna(), "_parse_error"] = "missing_github_repo"
df.loc[df["_commit_sha"].isna(), "_parse_error"] = "missing_commit_sha"
df.loc[df["_github_repo"].isna() & df["_commit_sha"].isna(), "_parse_error"] = "missing_repo_and_sha"

missing = df[df["_parse_error"].notna()]
print(f"Parsed {len(df) - len(missing):,} commit references")
print(f"Rows missing repo or SHA: {len(missing):,}")
display(df[[COMMIT_COLUMN, "_github_repo", "_commit_sha", "_commit_url"]].head(10))

Parsed 3,312 commit references
Rows missing repo or SHA: 0


,COMMIT,_github_repo,_commit_sha,_commit_url
0,https://github.com/dbmdz/digitalcollections-mo...,dbmdz/digitalcollections-model,57eb5bcbd60a983f1b80aa09589a51c9ae69cead,https://github.com/dbmdz/digitalcollections-mo...
1,https://github.com/btheu/estivate/commit/22adf...,btheu/estivate,22adfa9009bf25c55b5249a247e9057dc0233505,https://github.com/btheu/estivate/commit/22adf...
2,https://github.com/jcabi/jcabi-http/commit/1a2...,jcabi/jcabi-http,1a244d71abdfb74301de50affd188e52768d04c5,https://github.com/jcabi/jcabi-http/commit/1a2...
3,https://github.com/ashwanthkumar/gocd-java-cli...,ashwanthkumar/gocd-java-client,560d244d163210e7fc2bf80a6845376254c35b1d,https://github.com/ashwanthkumar/gocd-java-cli...
4,https://github.com/grasshopper7/pdfextentrepor...,grasshopper7/pdfextentreporter,bae2baa9c0254191809429b4dcb1c3fbdaaffc58,https://github.com/grasshopper7/pdfextentrepor...
5,https://github.com/kuehne-trustable-de/ca3sCor...,kuehne-trustable-de/ca3sCore,3e1623e622e1fe522b06247ca01105e7987c3ca9,https://github.com/kuehne-trustable-de/ca3sCor...
6,https://github.com/jeremylong/DependencyCheck/...,jeremylong/DependencyCheck,b7dd86d403c6238933ad7f75a3d12848e08c2d3f,https://github.com/jeremylong/DependencyCheck/...
7,https://github.com/appNG/appng/commit/4492c722...,appNG/appng,4492c722675743f59decc2c323444a34002edb88,https://github.com/appNG/appng/commit/4492c722...
8,https://github.com/s-frei/TrackSearch/commit/2...,s-frei/TrackSearch,27996d96e31c55c5cb554e21947636cf294530ea,https://github.com/s-frei/TrackSearch/commit/2...
9,https://github.com/Nasdanika/core/commit/386f2...,Nasdanika/core,386f292d5cf08bc950a3287b5213f3edf0ccfe83,https://github.com/Nasdanika/core/commit/386f2...


In [11]:
def load_cache(path):
    if not path.exists():
        return {}
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def save_cache(path, cache):
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    with tmp_path.open("w", encoding="utf-8") as f:
        json.dump(cache, f, indent=2, sort_keys=True)
    tmp_path.replace(path)


def normalize_token(token):
    token = (token or "").strip()
    if token.lower().startswith("bearer "):
        token = token.split(None, 1)[1].strip()
    return token


def github_headers(token):
    token = normalize_token(token)
    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
    }
    if token:
        headers["Authorization"] = f"Bearer {token}"
    return headers


def seconds_until_rate_limit_reset(response):
    reset_epoch = int(response.headers.get("X-RateLimit-Reset", "0") or "0")
    if reset_epoch <= 0:
        return REQUEST_SLEEP_SECONDS * 10
    return max(0, reset_epoch - time.time()) + RATE_LIMIT_BUFFER_SECONDS


def web_headers():
    return {
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "User-Agent": "Mozilla/5.0 commit-date-miner",
    }


def normalize_datetime(value):
    if not value:
        return None
    try:
        return pd.to_datetime(value, utc=True).isoformat().replace("+00:00", "Z")
    except Exception:
        return None


def request_get(session, url, headers):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.get(url, headers=headers, timeout=REQUEST_TIMEOUT_SECONDS)
        except requests.RequestException as exc:
            if attempt == MAX_RETRIES:
                return None, f"request_error:{type(exc).__name__}"
            time.sleep(REQUEST_SLEEP_SECONDS * attempt)
            continue

        if response.status_code == 403 and response.headers.get("X-RateLimit-Remaining") == "0":
            wait_seconds = seconds_until_rate_limit_reset(response)
            reset_time = datetime.fromtimestamp(time.time() + wait_seconds, tz=timezone.utc).isoformat()
            print(f"Rate limit reached. Sleeping until about {reset_time} UTC.")
            time.sleep(wait_seconds)
            continue

        if response.status_code in {429, 500, 502, 503, 504} and attempt < MAX_RETRIES:
            time.sleep(REQUEST_SLEEP_SECONDS * attempt)
            continue

        return response, None

    return None, "max_retries_exceeded"


def extract_datetime_from_commit_html(html):
    patterns = [
        r"<(?:relative-time|time-ago)\\b[^>]*datetime=\"([^\"]+)\"",
        r"datetime=\"([^\"]+)\"",
    ]
    for pattern in patterns:
        for match in re.finditer(pattern, html, flags=re.IGNORECASE):
            commit_date = normalize_datetime(match.group(1))
            if commit_date:
                return commit_date
    return None


def extract_datetime_from_patch(patch_text):
    match = re.search(r"^Date:\\s*(.+)$", patch_text, flags=re.MULTILINE)
    if not match:
        return None
    try:
        parsed = parsedate_to_datetime(match.group(1))
    except Exception:
        return None
    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=timezone.utc)
    return parsed.astimezone(timezone.utc).isoformat().replace("+00:00", "Z")


def fetch_commit_date_from_page(session, commit_url):
    response, error = request_get(session, commit_url, web_headers())
    if error:
        return None, error
    if response.status_code == 404:
        return None, "404_not_found"
    if response.status_code == 401:
        return None, "http_401_private_or_login_required"
    if not response.ok:
        return None, f"http_{response.status_code}"

    commit_date = extract_datetime_from_commit_html(response.text)
    if commit_date:
        return commit_date, None

    patch_url = commit_url.rstrip("/") + ".patch"
    patch_response, patch_error = request_get(session, patch_url, web_headers())
    if patch_error:
        return None, patch_error
    if patch_response.ok:
        commit_date = extract_datetime_from_patch(patch_response.text)
        if commit_date:
            return commit_date, None

    return None, "missing_date_in_commit_page"


def fetch_commit_date_from_api(session, repo, sha, token):
    url = f"https://api.github.com/repos/{repo}/commits/{sha}"
    response, error = request_get(session, url, github_headers(token))
    if error:
        return None, error
    if response.status_code == 401:
        return None, "http_401_bad_or_expired_token"
    if response.status_code == 404:
        return None, "404_not_found"
    if not response.ok:
        return None, f"http_{response.status_code}"

    data = response.json()
    commit_date = data.get("commit", {}).get("committer", {}).get("date")
    if not commit_date:
        commit_date = data.get("commit", {}).get("author", {}).get("date")
    return normalize_datetime(commit_date), None if commit_date else "missing_date_in_api_response"


def fetch_commit_date(session, repo, sha, commit_url, token):
    if commit_url:
        commit_date, error = fetch_commit_date_from_page(session, commit_url)
        if commit_date or not normalize_token(token):
            return commit_date, error

    if repo and sha and normalize_token(token):
        return fetch_commit_date_from_api(session, repo, sha, token)

    return None, "missing_commit_url_or_private_commit"

In [12]:
cache = load_cache(CACHE_JSON)
session = requests.Session()

unique_refs = (
    df[["_github_repo", "_commit_sha", "_commit_url"]]
    .dropna(subset=["_commit_url"])
    .drop_duplicates()
    .itertuples(index=False, name=None)
)
unique_refs = list(unique_refs)

def cache_key(repo, sha, commit_url):
    return commit_url or f"{repo}@{sha}"


def should_request(repo, sha, commit_url):
    entry = cache.get(cache_key(repo, sha, commit_url))
    if entry is None:
        return True
    if entry.get("commit_date"):
        return False
    return RETRY_FAILED_CACHE


cached_success_count = sum(
    bool(cache.get(cache_key(repo, sha, commit_url), {}).get("commit_date"))
    for repo, sha, commit_url in unique_refs
)
cached_failure_count = sum(
    cache_key(repo, sha, commit_url) in cache and not cache.get(cache_key(repo, sha, commit_url), {}).get("commit_date")
    for repo, sha, commit_url in unique_refs
)
refs_to_mine = [(repo, sha, commit_url) for repo, sha, commit_url in unique_refs if should_request(repo, sha, commit_url)]

print(f"Unique GitHub commits: {len(unique_refs):,}")
print(f"Cached successful dates: {cached_success_count:,}")
print(f"Cached failures: {cached_failure_count:,}")
print(f"Need to request from GitHub: {len(refs_to_mine):,}")

if not normalize_token(GITHUB_TOKEN):
    print("No GitHub token provided. Public commits may work, but the rate limit will be much lower.")

for index, (repo, sha, commit_url) in enumerate(tqdm(refs_to_mine, desc="Mining commit dates", unit="commit"), start=1):
    key = cache_key(repo, sha, commit_url)

    commit_date, error = fetch_commit_date(session, repo, sha, commit_url, GITHUB_TOKEN)

    cache[key] = {
        "repo": repo,
        "sha": sha,
        "commit_url": commit_url,
        "commit_date": commit_date,
        "error": error,
        "mined_at": datetime.now(timezone.utc).isoformat(),
    }

    if index % 25 == 0:
        save_cache(CACHE_JSON, cache)
        print(f"Requested {index:,}/{len(refs_to_mine):,}")

    time.sleep(REQUEST_SLEEP_SECONDS)

save_cache(CACHE_JSON, cache)
print(f"Cache saved to {CACHE_JSON}")

Unique GitHub commits: 1,608
Cached successful dates: 0
Cached failures: 0
Need to request from GitHub: 1,608


Mining commit dates:   0%|          | 0/1608 [00:00<?, ?commit/s]

Requested 25/1,608
Requested 50/1,608
Requested 75/1,608
Requested 100/1,608
Requested 125/1,608
Requested 150/1,608
Requested 175/1,608
Requested 200/1,608
Requested 225/1,608
Requested 250/1,608
Requested 275/1,608
Requested 300/1,608
Requested 325/1,608
Requested 350/1,608
Requested 375/1,608
Requested 400/1,608
Requested 425/1,608
Requested 450/1,608
Requested 475/1,608
Requested 500/1,608
Requested 525/1,608
Requested 550/1,608
Requested 575/1,608
Requested 600/1,608
Requested 625/1,608
Requested 650/1,608
Requested 675/1,608
Requested 700/1,608
Requested 725/1,608
Requested 750/1,608
Requested 775/1,608
Requested 800/1,608
Requested 825/1,608
Requested 850/1,608
Requested 875/1,608
Requested 900/1,608
Requested 925/1,608
Requested 950/1,608
Requested 975/1,608
Requested 1,000/1,608
Requested 1,025/1,608
Requested 1,050/1,608
Requested 1,075/1,608
Requested 1,100/1,608
Requested 1,125/1,608
Requested 1,150/1,608
Requested 1,175/1,608
Requested 1,200/1,608
Requested 1,225/1,608
Req

In [13]:
def cache_lookup(row, field):
    repo = row["_github_repo"]
    sha = row["_commit_sha"]
    commit_url = row["_commit_url"]
    if pd.isna(commit_url):
        return None
    return cache.get(cache_key(repo, sha, commit_url), {}).get(field)


df[DATE_COLUMN] = df.apply(lambda row: cache_lookup(row, "commit_date"), axis=1)
df["COMMIT_DATE_ERROR"] = df.apply(lambda row: cache_lookup(row, "error"), axis=1)
df["COMMIT_DATE_ERROR"] = df["_parse_error"].fillna(df["COMMIT_DATE_ERROR"])

# Keep helper columns for auditing. Drop these two lines if you do not want them in the output CSV.
df["GITHUB_REPO_PARSED"] = df["_github_repo"]
df["COMMIT_SHA_PARSED"] = df["_commit_sha"]
df["COMMIT_URL_PARSED"] = df["_commit_url"]

output_df = df.drop(columns=["_github_repo", "_commit_sha", "_commit_url", "_parse_error"])
failure_df = output_df[output_df[DATE_COLUMN].isna() | output_df["COMMIT_DATE_ERROR"].notna()].copy()
output_df.to_csv(OUTPUT_CSV, index=False)
failure_df.to_csv(FAILURES_CSV, index=False)

print(f"Rows with commit date: {output_df[DATE_COLUMN].notna().sum():,}/{len(output_df):,}")
print(f"Wrote {OUTPUT_CSV.resolve()}")
print(f"Failed rows: {len(failure_df):,}")
print(f"Wrote {FAILURES_CSV.resolve()}")
display(output_df.head())

Rows with commit date: 3,248/3,312
Wrote ../../data/baseline/RESPONSE_with_commit_dates.csv
Failed rows: 64
Wrote ../../data/baseline/RESPONSE_commit_date_failures.csv


,CVE,UPSTREAM G:A:V,DOWNSTREAM G:A:V,DOWNSTREAM REPO,COMMIT,COMMIT_DATE,COMMIT_DATE_ERROR,GITHUB_REPO_PARSED,COMMIT_SHA_PARSED,COMMIT_URL_PARSED
0,CVE-2021-39154,com.thoughtworks.xstream:xstream,ne,dbmdz/digitalcollections-model,https://github.com/dbmdz/digitalcollections-mo...,2020-11-18T08:28:46Z,None,dbmdz/digitalcollections-model,57eb5bcbd60a983f1b80aa09589a51c9ae69cead,https://github.com/dbmdz/digitalcollections-mo...
1,CVE-2021-37714,org.jsoup:jsoup,com.github.btheu.estivate:estivate,btheu/estivate,https://github.com/btheu/estivate/commit/22adf...,2021-12-03T15:44:15Z,None,btheu/estivate,22adfa9009bf25c55b5249a247e9057dc0233505,https://github.com/btheu/estivate/commit/22adf...
2,CVE-2021-37714,org.jsoup:jsoup,com.jcabi:jcabi-http,jcabi/jcabi-http,https://github.com/jcabi/jcabi-http/commit/1a2...,2021-08-16T23:00:43Z,None,jcabi/jcabi-http,1a244d71abdfb74301de50affd188e52768d04c5,https://github.com/jcabi/jcabi-http/commit/1a2...
3,CVE-2021-37714,org.jsoup:jsoup,in.ashwanthkumar:gocd-java-client,ashwanthkumar/gocd-java-client,https://github.com/ashwanthkumar/gocd-java-cli...,2021-09-30T23:03:50Z,None,ashwanthkumar/gocd-java-client,560d244d163210e7fc2bf80a6845376254c35b1d,https://github.com/ashwanthkumar/gocd-java-cli...
4,CVE-2021-37714,org.jsoup:jsoup,tech.grasshopper:pdfextentreporter,grasshopper7/pdfextentreporter,https://github.com/grasshopper7/pdfextentrepor...,2022-03-01T12:31:07Z,None,grasshopper7/pdfextentreporter,bae2baa9c0254191809429b4dcb1c3fbdaaffc58,https://github.com/grasshopper7/pdfextentrepor...


In [14]:
error_summary = (
    output_df["COMMIT_DATE_ERROR"]
    .fillna("no_error")
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="rows")
)

display(error_summary)

if len(failure_df):
    print(f"Failure rows were written to {FAILURES_CSV.resolve()}")
    display(failure_df.head(20))

if len(missing):
    print("Rows that could not be parsed. If COMMIT only has a SHA, set REPO_COLUMN in the first cell.")
    display(missing[[COMMIT_COLUMN, "_github_repo", "_commit_sha", "_commit_url"]].head(20))

,status,rows
0,no_error,3248
1,404_not_found,64


Failure rows were written to ../../data/baseline/RESPONSE_commit_date_failures.csv


,CVE,UPSTREAM G:A:V,DOWNSTREAM G:A:V,DOWNSTREAM REPO,COMMIT,COMMIT_DATE,COMMIT_DATE_ERROR,GITHUB_REPO_PARSED,COMMIT_SHA_PARSED,COMMIT_URL_PARSED
50,CVE-2019-3888,io.undertow:undertow-core,com.networknt:oauth2-cache,networknt/light-oauth2,https://github.com/networknt/light-oauth2/comm...,None,404_not_found,networknt/light-oauth2,62be256827b5e22f7c66b1faf8f8193284614720,https://github.com/networknt/light-oauth2/comm...
51,CVE-2019-3888,io.undertow:undertow-core,com.networknt:eventuate-cdc-service,networknt/light-eventuate-4j,https://github.com/networknt/light-eventuate-4...,None,404_not_found,networknt/light-eventuate-4j,c6464e31c63a57cab041a300a1a80af8afea5e0a,https://github.com/networknt/light-eventuate-4...
61,CVE-2019-3888,io.undertow:undertow-core,com.networknt:oauth2-cache,networknt/light-oauth2,https://github.com/networknt/light-oauth2/comm...,None,404_not_found,networknt/light-oauth2,62be256827b5e22f7c66b1faf8f8193284614720,https://github.com/networknt/light-oauth2/comm...
62,CVE-2019-3888,io.undertow:undertow-core,com.networknt:eventuate-cdc-service,networknt/light-eventuate-4j,https://github.com/networknt/light-eventuate-4...,None,404_not_found,networknt/light-eventuate-4j,c6464e31c63a57cab041a300a1a80af8afea5e0a,https://github.com/networknt/light-eventuate-4...
153,CVE-2017-12196,io.undertow:undertow-core,com.networknt:oauth2-cache,networknt/light-oauth2,https://github.com/networknt/light-oauth2/comm...,None,404_not_found,networknt/light-oauth2,78368e5652ef7205a05c42e699df1b6e38764b4e,https://github.com/networknt/light-oauth2/comm...
161,CVE-2017-12196,io.undertow:undertow-core,com.networknt:cdc-service,networknt/light-eventuate-4j,https://github.com/networknt/light-eventuate-4...,None,404_not_found,networknt/light-eventuate-4j,9b88e486fb808da093f130105f7f1821d10ef181,https://github.com/networknt/light-eventuate-4...
163,CVE-2017-12196,io.undertow:undertow-core,com.networknt:oauth2-cache,networknt/light-oauth2,https://github.com/networknt/light-oauth2/comm...,None,404_not_found,networknt/light-oauth2,78368e5652ef7205a05c42e699df1b6e38764b4e,https://github.com/networknt/light-oauth2/comm...
171,CVE-2017-12196,io.undertow:undertow-core,com.networknt:cdc-service,networknt/light-eventuate-4j,https://github.com/networknt/light-eventuate-4...,None,404_not_found,networknt/light-eventuate-4j,9b88e486fb808da093f130105f7f1821d10ef181,https://github.com/networknt/light-eventuate-4...
228,CVE-2019-0201,org.apache.zookeeper:zookeeper,com.salesforce.argus:argus-core,salesforce/Argus,https://github.com/salesforce/Argus/commit/585...,None,404_not_found,salesforce/Argus,5854e08b6f166e25e1c11740477269efa1fae1f7,https://github.com/salesforce/Argus/commit/585...
307,CVE-2018-11039,org.springframework:spring-web,com.assist4j:assist4j-core,yuweihn/assist4j,https://github.com/yuweihn/assist4j/commit/6a4...,None,404_not_found,yuweihn/assist4j,6a4ea64a758d0cbf0274f3f6240360098c9a62f5,https://github.com/yuweihn/assist4j/commit/6a4...
